In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*See the [API reference](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่มีเฉพาะสำหรับผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น ยังอยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>
## ภาพรวม
ในเคมีควอนตัม ปัญหาโครงสร้างอิเล็กตรอนเน้นที่การหาคำตอบของสมการ Schrödinger อิเล็กทรอนิกส์ ซึ่งเป็นฟังก์ชันคลื่นควอนตัมที่อธิบายพฤติกรรมของอิเล็กตรอนในระบบ ฟังก์ชันคลื่นเหล่านี้เป็นเวกเตอร์ของแอมพลิจูดเชิงซ้อน โดยแต่ละแอมพลิจูดสอดคล้องกับการมีส่วนร่วมของการจัดวางอิเล็กตรอนที่เป็นไปได้

ground state คือฟังก์ชันคลื่นที่มีพลังงานต่ำสุดของระบบ และมีความสำคัญพิเศษในการศึกษาระบบโมเลกุล วิธีการที่แม่นยำที่สุดในการคำนวณ ground state พิจารณาการจัดวางอิเล็กตรอนทั้งหมดที่เป็นไปได้ แต่วิธีนี้ไม่สามารถทำได้จริงสำหรับระบบขนาดใหญ่ เนื่องจากจำนวนการจัดวางเพิ่มขึ้นแบบ exponential ตามขนาดของระบบ

Handover Iterative Variational Quantum Eigensolver (HI-VQE) เป็นวิธีไฮบริดควอนตัม-คลาสสิกที่ล้ำสมัยสำหรับการประมาณ ground state ของระบบโมเลกุลอย่างแม่นยำ ผสานรวม quantum hardware กับการคำนวณแบบคลาสสิก โดยใช้ quantum processor เพื่อสำรวจการจัดวางอิเล็กตรอนที่เป็นผู้สมัครอย่างมีประสิทธิภาพ และคำนวณฟังก์ชันคลื่นผลลัพธ์บนคอมพิวเตอร์คลาสสิก ด้วยการสร้างฟังก์ชันคลื่นที่กระชับแต่มีความแม่นยำทางเคมี HI-VQE ช่วยส่งเสริมการวิจัยและการค้นพบในเคมีควอนตัมและวิทยาศาสตร์วัสดุ

![ภาพแสดงภาพรวมของอัลกอริทึม HI-VQE ของ Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE ลดความซับซ้อนทางการคำนวณของปัญหาโครงสร้างอิเล็กตรอนด้วยการประมาณ ground state อย่างมีประสิทธิภาพด้วยความแม่นยำสูง มุ่งเน้นที่ชุดย่อยของการจัดวางอิเล็กตรอนที่เกี่ยวข้องมากที่สุดที่คัดเลือกมาอย่างดี เพื่อเพิ่มประสิทธิภาพทั้งความแม่นยำและประสิทธิภาพการคำนวณ

ด้วยการรวมจุดแข็งของทั้งคอมพิวเตอร์คลาสสิกและควอนตัม HI-VQE ปรับปรุงและพัฒนาการประมาณฟังก์ชันคลื่นปัจจุบันแบบ iterative เทคนิคการสร้าง subspace เฉพาะตัวช่วยให้การเลือกการจัดวางมีประสิทธิภาพมากขึ้น ทำให้ผู้ใช้มีการควบคุมการคำนวณที่มากขึ้นและความแม่นยำที่ดีขึ้นในการจำลองเคมีควอนตัม

หากต้องการเรียนรู้เกี่ยวกับอัลกอริทึมในเชิงลึกมากขึ้น สามารถ [อ่านบทความวิจัยที่เกี่ยวข้อง](https://arxiv.org/abs/2503.06292) ได้
## คำอธิบาย
จำนวนการจัดวางอิเล็กตรอนสำหรับระบบโมเลกุลเพิ่มขึ้นแบบ exponential ตามขนาดระบบ อย่างไรก็ตาม สำหรับสถานะอิเล็กทรอนิกส์บางอย่าง เช่น ground state มักพบว่ามีเพียงส่วนเล็กน้อยของการจัดวางที่มีส่วนร่วมอย่างมีนัยสำคัญต่อพลังงานของสถานะนั้น วิธี Selected configuration interaction (SCI) ใช้ประโยชน์จากความกระจัดกระจายนี้เพื่อลดต้นทุนการคำนวณโดยการระบุและมุ่งเน้นที่การจัดวางที่เกี่ยวข้องมากที่สุด ชุดย่อยของการจัดวางนี้เรียกว่า subspace

HI-VQE ใช้ประสิทธิภาพโดยธรรมชาติของ quantum computer ในการแสดงแทนระบบโมเลกุลเพื่อช่วยในการค้นหา subspace ผสมผสาน subroutine คลาสสิกและควอนตัมเพื่อแก้ปัญหาโครงสร้างอิเล็กตรอนด้วยความแม่นยำสูง ต่างจากวิธี quantum SCI ที่มีอยู่ HI-VQE รวมการฝึก variational, การสร้าง subspace แบบ iterative และการคัดกรองการจัดวางแบบ pre-diagonalization เพื่อเพิ่มประสิทธิภาพด้วยการลดการวัดควอนตัม, การ iteration และต้นทุน diagonalization แบบคลาสสิก HI-VQE จึงสามารถนำไปใช้กับระบบโมเลกุลขนาดใหญ่ขึ้นที่ต้องการ Qubit มากขึ้น และลดต้นทุนในการแก้ปัญหาขนาดที่กำหนดด้วยระดับความแม่นยำเดิม

![ภาพแสดงคำอธิบายรายละเอียดของแต่ละขั้นตอนในอัลกอริทึม HI-VQE ของ Qunova](../docs/images/guides/qunova-chemistry/description.avif)

ในการคำนวณ ground state ของระบบ HI-VQE จะใช้แพ็กเกจเคมี PySCF แบบคลาสสิกก่อนเพื่อสร้างการแสดงแทนโมเลกุลจาก input ที่ผู้ใช้ระบุ เช่น ข้อมูลทางเรขาคณิตของโมเลกุลและข้อมูลโมเลกุลอื่น ๆ จากนั้นเข้าสู่ loop การ optimize แบบไฮบริดควอนตัม-คลาสสิก โดย iterative ปรับปรุง subspace เพื่อแสดงแทน ground state ได้อย่างเหมาะสมในขณะที่ลดจำนวนการจัดวางให้น้อยที่สุด loop จะดำเนินต่อไปจนกว่าจะตรงตามเกณฑ์การบรรจบ เช่น ขนาด subspace หรือความเสถียรของพลังงาน จากนั้นจึงส่งออกฟังก์ชันคลื่น ground state และพลังงานที่คำนวณได้ ผลลัพธ์เหล่านี้สามารถใช้สร้าง potential energy surface ที่แม่นยำและทำการวิเคราะห์เพิ่มเติมของระบบ

loop การ optimize มุ่งเน้นที่การปรับ parameter ของ quantum circuit เพื่อสร้าง subspace คุณภาพสูง HI-VQE มีตัวเลือก quantum circuit สามแบบ: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) และ [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html) การ optimize เริ่มต้นใกล้กับ reference state ของ Hartree-Fock เนื่องจากมีความเหมาะสมทั่วไป จากนั้น Circuit จะถูกประมวลผลบน quantum device และการจัดวางจะถูก sample จาก quantum state ผลลัพธ์ก่อนส่งกลับมาเป็น binary string เนื่องจาก noise ของ quantum device บางการจัดวางที่ sample มาอาจไม่ถูกต้องทางกายภาพ ล้มเหลวในการอนุรักษ์จำนวนอิเล็กตรอนหรือ spin HI-VQE แก้ปัญหานี้โดยใช้กระบวนการ configuration recovery จากแพ็กเกจ [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview) เพื่อให้ผู้ใช้สามารถแก้ไขการจัดวางที่ไม่ถูกต้องหรือละทิ้งได้

การจัดวางที่ถูกต้องจะผ่านขั้นตอนการคัดกรองเพิ่มเติมโดยเป็นทางเลือก เพื่อลบรายการที่คาดว่าจะมีส่วนร่วมน้อยที่สุด ซึ่งลดขนาดของ subspace ทำให้ลดต้นทุนของขั้นตอน diagonalization หากเปิดใช้งานการคัดกรอง จะสร้าง subspace Hamiltonian เบื้องต้นจากการจัดวางที่ถูกต้อง และทำ diagonalization ด้วยเกณฑ์การสิ้นสุดที่ผ่อนปรนมาก แม้ว่าความแม่นยำของ amplitude ผลลัพธ์สำหรับแต่ละการจัดวางจะต่ำ แต่มีประสิทธิภาพในการทำนายว่าการจัดวางใดควรถูกยกเว้นจาก subspace ใน iteration นี้ และคำนวณได้รวดเร็ว

การจัดวางที่เลือกจะถูกเพิ่มเข้าใน subspace และ Hamiltonian ของระบบจะถูก project เข้าสู่ subspace นี้ subspace อัปเดตแบบ iterative โดยเก็บรักษาการจัดวางที่เกี่ยวข้องมากที่สุดในแต่ละ iteration แนวทางนี้แตกต่างจากวิธีอื่นเพราะ quantum circuit ไม่จำเป็นต้องประมาณ ground state เต็มรูปแบบในแต่ละขั้นตอน

ต่อจากนั้น subspace Hamiltonian จะถูก diagonalize แบบคลาสสิกเพื่อหา eigenvalue ต่ำสุดและ eigenvector ที่สอดคล้อง ซึ่งแสดงการประมาณ ground state และพลังงานของมัน เมื่อคุณภาพ subspace ดีขึ้นผ่านการ iteration ground state ที่คำนวณได้จะประมาณ ground state จริงได้ดีขึ้น ขั้นตอนการคัดกรองเพิ่มเติมสามารถดำเนินการได้ ณ จุดนี้ เพื่อลบการจัดวางที่ไม่มีส่วนร่วมอย่างมีนัยสำคัญใน ground state ที่คำนวณได้ออกจาก subspace ขั้นตอนนี้ทำให้ subspace ที่นำเข้าสู่ iteration ถัดไปกระชับที่สุด โดยประเมินจาก amplitude ที่ได้จาก diagonalization เนื่องจากสิ่งเหล่านี้แสดงถึงความสำคัญของการมีส่วนร่วมของแต่ละการจัดวางต่อ ground state ที่คำนวณได้

การตรวจสอบการบรรจบจะกำหนดว่าการฝึกเพิ่มเติมจะปรับปรุงผลลัพธ์หรือไม่ ถ้าใช่ จะทำขั้นตอน classical expansion เพิ่มเติม parameter ของ quantum circuit จะถูกอัปเดตเพื่อลดพลังงานที่คำนวณได้เพิ่มเติม และกระบวนการจะวนซ้ำ ขั้นตอน classical expansion สร้างการจัดวางเพิ่มเติมสำหรับ subspace โดย supplement การจัดวางที่ sample จาก quantum device โดยระบุการจัดวางที่มี amplitude ใหญ่สุดในผลการ diagonalization ก่อน จากนั้นสร้างการจัดวางใหม่ด้วย single และ double excitation จากการจัดวางที่ระบุ แล้วนำการจัดวางตามจำนวนที่ต้องการเพิ่มเข้าใน subspace

เมื่อพิจารณาแล้วว่า iteration บรรจบแล้ว HI-VQE จะส่งคืน ground state ที่คำนวณได้ (ในรูปแบบของสถานะใน subspace และ amplitude ของสถานะเหล่านั้นในฟังก์ชันคลื่น ground state), พลังงานของมัน และค่าวัด energy variance ที่บอกว่าสถานะที่คำนวณได้เป็น eigenstate ของ Hamiltonian ของระบบหรือไม่

ผู้ใช้สามารถกำหนด quantum circuit ที่ใช้และจำนวน shot สำหรับแต่ละ quantum circuit รวมถึงควบคุมขนาด subspace หรือเปิดใช้งานการสร้างการจัดวางเพิ่มเติมแบบคลาสสิกเพื่อช่วย quantum generated configuration ในลักษณะนี้ผู้ใช้สามารถปรับพฤติกรรมของ HI-VQE ให้เหมาะสมกับแอปพลิเคชันที่ต้องการได้
## การอนุญาตสิทธิ์
โปรดทราบว่าการใช้งาน Qiskit Function นี้จำกัดเฉพาะปัญหาที่ต้องใช้ Qubit ไม่เกิน 20 ตัว เว้นแต่จะได้รับใบอนุญาตที่อนุญาตให้ใช้จำนวนสูงกว่า

หากต้องการสอบถามเกี่ยวกับการขอใบอนุญาต สามารถส่งอีเมลไปที่ [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com)
## เริ่มต้นใช้งาน
ขั้นแรก [ขอสิทธิ์เข้าถึง function](https://forms.office.com/r/zN3hvMTqJ1)
จากนั้น authenticate โดยใช้ [IBM Quantum&reg; API key](http://quantum.cloud.ibm.com/) และสมมติว่าคุณได้ [บันทึกบัญชีของคุณ](/guides/functions#install-qiskit-functions-catalog-client) ในสภาพแวดล้อมในเครื่องแล้ว เลือก Qiskit Function ดังนี้:

In [1]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load the function
function = catalog.load("qunova/hivqe-chemistry")

สามารถกำหนดและระบุตัวเลือกเพิ่มเติมสำหรับระบบโมเลกุลในรูปแบบ dictionary ต่อไปนี้

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

ประมวลผล function ด้วย input เรขาคณิตและตัวเลือก

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

ควร print Function job ID เพื่อให้สามารถใส่ใน support request ได้หากมีปัญหา

In [ ]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits,
    # for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

It is a good idea to print the Function job ID so that it can be provided in support requests if something goes wrong.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


ตัวอย่างนี้ใช้ 16 Qubit กับ 8 orbital ของ basis sto3g สำหรับโมเลกุล NH3
ตรวจสอบ[สถานะ](/guides/functions#check-job-status)ของ Qiskit Function workload หรือดึง[ผลลัพธ์](/guides/functions#retrieve-results)ดังนี้:

In [7]:
print(job.status())

QUEUED


After the job is completed, the results can be obtained with `result()` instance.

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


หลังจาก job เสร็จสิ้น สามารถรับผลลัพธ์ด้วย instance `result()`

In [ ]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: "
    f"{abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


## Performance

This section shows the demonstrated benchmark calculations of HI-VQE with a 24-qubit case for Li2S, a 40-qubit case for an N2 molecule, and a 44-qubit case for an FeP-NO system.

#### Dissociation potential energy surface curve for an Li2S molecule with 24 qubits

The PES curve is shown with the FCI reference and initial guess from RHF, together with the energy error from the FCI reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

The calculations have been conducted with the following geometries and options.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

The red dots represent the HI-VQE calculation results for six different geometries, and three geometries corresponding to 1.51, 2.40, and 3.80 Angstrom are provided as input in the above cell.

#### Dissociation PES curve for an N2 molecule with 40 qubits

The nitrogen molecule has been identified as a multireference system with large correlation energy contributions beyond the Hartree-Fock state. We conducted a benchmark calculation for the N2 molecule with cc-pvdz basis, (20o,14e) using the homo-lumo active orbital selection. The complete active space (CAS) number to represent this problem is 6,009,350,400. It is not possible to obtain the eigenvalue problem solution (for energy and electronic structure) with this number of states using a powerful desktop (16cpu/64GB). With HI-VQE, users can efficiently search the subspace of CAS states to find chemically accurate results while saving computation resources significantly. The following plots show the PES curve of 40 qubits HI-VQE calculation of the N2 molecule dissociation.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the N2 system](../docs/images/guides/qunova-chemistry/N2_PES_40qubits.avif)

#### Dissociation PES curve for five-coordinated iron(II)-porphyrin with an NO system with 44 qubits

Another interesting chemical system is an iron(II)-porphyrin (FeP) complex with a coordinated nitric oxide (NO) ligand, which represents a biologically relevant metalloporphyrin system that plays crucial roles in various physiological processes. In this example, HI-VQE has been utilized to estimate the accurate potential energy surface curve of the intermolecular interaction between FeP and NO (ground state energy for differently separated geometries). The combined system has 450 orbitals and 202 electrons (450o,202e) with 6-31g(d) basis in total. The homo-lumo active orbital selection was utilized to calculate the smaller case from the real case with (22o,22e). From the following benchmark results, we were able to achieve the chemical accuracy (>&nbsp;1.6 mHa) with a state-of-the-art classical computer chemistry calculation of CASCI(DMRG) (22o,22e) reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the FeP-NO system](../docs/images/guides/qunova-chemistry/fepno_44qubits.avif)

## Benchmarks

- Exact matrix size is the number of determinants for exact solution, such as FCI and CASCI.
- HI-VQE calculation samples and calculates the subspace of it (as in, HI-VQE matrix size).
- Total time includes QPU runtime and Qiskit Function runs with CPU.
- Accuracy is estimated from the energy difference from exact solution.

|   Chemical system  | Number of qubits | Exact matrix size |  HI-VQE matrix size  | E(diff) from exact (mHa) | Number of iteration | Total time    | QPU runtime usage |
| ------------------ | ---------------- | ----------------- | ------------------- | ------------------------ | ------------------- | ------------- | ----------------- |
|  $NH_3$ (8o,10e)   |  16              |  3136             |  1936               |  0.08                    |  6                  | 37 s          | 34 s              |
|  $Li_2S$ (10o,10e) |  20              |  63504            |  3969               |  0.60                    |  5                  | 250 s         | 50 s              |
|  $NH_3$ (15o,10e)  |  30              |  9018009          |  49729              |  0.90                    |  5                  | 354 s         | 54 s              |
|  $N_2$ (16o,14e)   |  32              |  130873600        |  1798281            |  1.10                    |  9                  | 6531 s        | 121 s             |
|  $3H_2O$ (18o,24e) |  36              |  344622096        |  399424             |  0.90                    |  24                 | 5174 s        | 130 s             |
|  $N_2$ (20o,14e)   |  40              |  6009350400       |  9012004            |  1.20                    |  21                 | 46547 s       | 258 s             |

## Fetch error messages

If your workload fails, the status will be `ERROR` and calling `job.result()` will raise an exception:

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Get support

You can send an email to [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) for help with this function.

If you want help with troubleshooting a specific error, please provide the Function job ID of the job that encountered the error.

## Next steps

<Admonition type="tip" title="Recommendations">
- Request access to the function by filling in [this form](https://forms.office.com/r/zN3hvMTqJ1).
- Visit the [API reference](/docs/api/functions/qunova-chemistry) for this Qiskit Function.
- Try the [Compute dissociation PES curve for FeP-NO with HI-VQE](/docs/tutorials/qunova-hivqe) tutorial.
- Review [Pellow-Jarman, A., et al. (2025).  HIVQE: Handover Iterative Variational Quantum Eigensolver for Efficient Quantum Chemistry Calculations. arXiv preprint arXiv:2503.06292](https://arxiv.org/abs/2503.06292).
- Try the [Dissociation PES curves with Qunova HiVQE](/docs/tutorials/qunova-hivqe) tutorial.
</Admonition>